In [2]:
!cd /content/MINI_PROJECT && git config user.name "Skm48"
!cd /content/MINI_PROJECT && git config user.email "skm48@student.le.ac.uk"


In [4]:
!git config pull.rebase false  # merge (the default strategy)
!git config pull.rebase true   # rebase
!git config pull.ff only       # fast-forward only

In [7]:
GITHUB_USER = 'Skm48'   # GitHub username
REPO_NAME   = 'MINI_PROJECT'

import os

if not os.path.exists(f'/content/{REPO_NAME}'):
    # First time — clone
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    print('Repo cloned!')
else:
    # Already cloned — pull latest
    !cd /content/{REPO_NAME} && git pull
    print('Pulled latest changes!')

os.chdir(f'/content/{REPO_NAME}')
print(f'Working directory: {os.getcwd()}')

fatal: Not possible to fast-forward, aborting.
Pulled latest changes!
Working directory: /content/MINI_PROJECT


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%cd MINI_PROJECT

/content/MINI_PROJECT/MINI_PROJECT


In [10]:
!cp /content/drive/MyDrive/Miniproject/archive.zip /content/MINI_PROJECT/data/
!unzip -q /content/MINI_PROJECT/data/archive.zip -d /content/MINI_PROJECT/data/

!ls /content/MINI_PROJECT/data/chest_xray/

replace /content/MINI_PROJECT/data/chest_xray/test/NORMAL/IM-0001-0001.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: chest_xray  __MACOSX  test  train  val


In [ ]:
# Remove the Mac junk folder and nested duplicate
!rm -rf /content/MINI_PROJECT/data/chest_xray/__MACOSX
!rm -rf /content/MINI_PROJECT/data/chest_xray/chest_xray

# Check it's clean
!ls /content/MINI_PROJECT/data/chest_xray/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
GITHUB_USER = 'Skm48'   # GitHub username
REPO_NAME   = 'MINI_PROJECT'

import os

if not os.path.exists(f'/content/{REPO_NAME}'):
    # First time — clone
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    print('Repo cloned!')
else:
    # Already cloned — pull latest
    !cd /content/{REPO_NAME} && git pull
    print('Pulled latest changes!')

os.chdir(f'/content/{REPO_NAME}')
print(f'Working directory: {os.getcwd()}')

In [ ]:

import os
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [ ]:
def collect_image_paths(raw_dir: str) -> pd.DataFrame:
    records = []
    label_map = {"NORMAL": 0, "PNEUMONIA": 1}

    for split in ["train", "val", "test"]:
        split_dir = os.path.join(raw_dir, split)
        if not os.path.exists(split_dir):
            print(f"Warning: {split_dir} not found, skipping")
            continue

        for class_name in ["NORMAL", "PNEUMONIA"]:
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                continue

            for fname in os.listdir(class_dir):
                if fname.lower().endswith((".jpeg", ".jpg", ".png")):
                    records.append({
                        "filepath": os.path.join(class_dir, fname),
                        "label": label_map[class_name],
                        "original_split": split,
                    })

    df = pd.DataFrame(records)
    print(f"Total images found: {len(df)}")
    print(f"  By original split: {dict(df['original_split'].value_counts())}")
    print(f"  By label: {dict(df['label'].value_counts())}")
    return df

In [ ]:
df = collect_image_paths("data/chest_xray")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

def stratified_split(df, seed=42, save_path=None):
    # Keep original test set untouched
    test_df = df[df["original_split"] == "test"].copy()

    # Merge train + val (val is only 16 images — useless)
    trainval_df = df[df["original_split"].isin(["train", "val"])].copy()

    print(f"Merged train+val: {len(trainval_df)} images")
    print(f"Original test kept: {len(test_df)} images")

    # Split trainval into new train (80%) and val (10%)
    # val_ratio = 0.1 / (0.8 + 0.1) ≈ 0.111
    train_df, val_df = train_test_split(
        trainval_df,
        test_size=0.111,
        stratify=trainval_df["label"],
        random_state=seed,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"\nNew split:")
    print(f"  Train: {len(train_df)}  (Normal: {sum(train_df['label']==0)}, Pneumonia: {sum(train_df['label']==1)})")
    print(f"  Val:   {len(val_df)}  (Normal: {sum(val_df['label']==0)}, Pneumonia: {sum(val_df['label']==1)})")
    print(f"  Test:  {len(test_df)}  (Normal: {sum(test_df['label']==0)}, Pneumonia: {sum(test_df['label']==1)})")

    # Save for reproducibility
    if save_path:
        train_df["split"] = "train"
        val_df["split"] = "val"
        test_df["split"] = "test"
        all_splits = pd.concat([train_df, val_df, test_df], ignore_index=True)
        all_splits.to_csv(save_path, index=False)
        print(f"Split indices saved to: {save_path}")
        train_df = train_df.drop(columns=["split"])
        val_df = val_df.drop(columns=["split"])
        test_df = test_df.drop(columns=["split"])

    return train_df, val_df, test_df

In [ ]:
train_df, val_df, test_df = stratified_split(df, save_path="data/split_indices.csv")

In [ ]:
from torchvision import transforms

def get_transforms(image_size=224):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    return {"train": train_transform, "val": eval_transform, "test": eval_transform}


In [ ]:
from PIL import Image

tfms = get_transforms()
img = Image.open(train_df.iloc[0]["filepath"]).convert("RGB")
tensor = tfms["train"](img)
print(f"Original size: {img.size}")
print(f"Tensor shape: {tensor.shape}")
print(f"Pixel range: [{tensor.min():.2f}, {tensor.max():.2f}]")

In [ ]:
class ChestXrayDataset(Dataset):
    """
    PyTorch Dataset for chest X-ray images.

    Args:
        dataframe: DataFrame with 'filepath' and 'label' columns
        transform: torchvision transform to apply
    """

    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row["label"], dtype=torch.long)
        return image, label

In [ ]:
train_ds = ChestXrayDataset(train_df, transform=tfms["train"])
img, label = train_ds[0]
print(f"Dataset size: {len(train_ds)}")
print(f"Image shape: {img.shape}")
print(f"Label: {label}")

In [ ]:
def compute_class_weights(train_df):
    counts = Counter(train_df["label"].values)
    total = sum(counts.values())
    weights = [total / (len(counts) * counts[i]) for i in range(len(counts))]
    weights_tensor = torch.FloatTensor(weights)
    print(f"Class weights: Normal={weights[0]:.3f}, Pneumonia={weights[1]:.3f}")
    return weights_tensor
compute_class_weights(train_df)

In [ ]:
!cd /content/MINI_PROJECT && git config user.name "Skm48"
!cd /content/MINI_PROJECT && git config user.email "skm48@student.le.ac.uk"


In [ ]:
!cd /content/MINI_PROJECT && git remote set-url origin https://Skm48@github.com/Skm48/MINI_PROJECT.git

In [ ]:
!cp /content/drive/MyDrive/Minihealthproject.ipynb /content/MINI_PROJECT/notebooks/
